# Week 4b.1: Messages as State

Before we rebuild the Husky Tech support agent as a graph, we need to revisit three things:
- what a **message** is
- what happens when a model is given **tools**, and 
- how a conversation survives inside graph **state**.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."
print("API key loaded")

## 0. Model Setup

Execute one of the following to define the model.

If using Ollama, you will need to start it first (simply open the chat UI and send a message):
- https://docs.langchain.com/oss/python/integrations/chat/ollama

Verify that Ollama is serving a model:
- http://localhost:11434/

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:4b", reasoning=False)

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

## 1. Messages

A conversation is a **list of messages**, and each message has a type that says who
produced it:

| Type | Who |
|---|---|
| `SystemMessage` | the instructions we set, never shown to the customer |
| `HumanMessage` | the customer |
| `AIMessage` | the model — either an answer, or a request to call a tool |
| `ToolMessage` | the result of a tool we ran on the model's behalf |

Nothing more than that. Build one by hand and look at it:

In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

conversation = [
    SystemMessage(content="You are a support assistant for Husky Tech."),
    HumanMessage(content="Has my order HT-1001 shipped?"),
    AIMessage(content="Let me check that for you."),
]

for m in conversation:
    m.pretty_print()

## 2. A model that can call tools

`bind_tools` hands the model the name, the signature and the docstring of each tool.
That is all it ever sees of them. It cannot run anything — it can only **ask**.

We will use the tools you wrote in W03b. They live in `support_tools.py` so this
notebook can stay about messages and state:

In [ ]:
from support_tools import look_up_order, check_return_eligibility, search_faq, escalate_to_human

tools = [look_up_order, check_return_eligibility, search_faq, escalate_to_human]
model_with_tools = model.bind_tools(tools)

Ask it something that needs no tool, and it answers in words:

In [ ]:
reply = model_with_tools.invoke([HumanMessage(content="Hello, who are you?")])
print(reply.text)
print("tool calls:", reply.tool_calls)

Now ask something it cannot answer without data:

In [ ]:
reply = model_with_tools.invoke([HumanMessage(content="Has order HT-1001 been delivered?")])
print("text:      ", repr(reply.text))
print("tool calls:", reply.tool_calls)

Note that the content is **empty**, and `tool_calls` holds a request:
the name of a tool and the arguments to call it with.

The model did not look anything up. It cannot. It produced a *request*, and something
else has to notice it, run the function, and hand the result back. 

## 3. Messages as state

A graph carries **state**: a dictionary every node can read, and that each node
updates by returning the keys it changed.

The obvious way to hold a conversation is a list of messages:

In [ ]:
from typing_extensions import TypedDict
from langchain.messages import AnyMessage

class ChatState(TypedDict):
    messages: list[AnyMessage]

There is an inherent problem with this approach, and it is easier to see than to describe. 

Let's build a two-node graph where each node returns a message, and watch what survives:

In [ ]:
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

def first(state: ChatState) -> dict:
    return {"messages": [AIMessage(content="FIRST node spoke")]}

def second(state: ChatState) -> dict:
    return {"messages": [AIMessage(content="SECOND node spoke")]}

# TODO: build a sequantial graph START -> first -> second -> END



graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
result = graph.invoke({"messages": [HumanMessage(content="hello")]})

for m in result["messages"]:
    print(f"{type(m).__name__:14s} {m.text}")

Only the last one is left. The customer's message and the first node's reply are
both gone.

That is the default behaviour of a state key: a node returns `messages`, and the new
value **replaces** the old one. For a key like `reply` that is exactly right. For a
conversation it is fatal because every turn would erase the history.

## 4. Reducers

A **reducer** is a rule attached to one key saying how an update combines with what
is already stored. 

We can use Annotated to attach an add_messages function to the messages key. LangGraph then applies the rule instead of overwriting.

`add_messages` is the prebuilt reducer for conversations: it appends.

- https://docs.langchain.com/oss/python/langgraph/graph-api#reducers

In [ ]:
from typing import Annotated
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

Let's take a look at the reducer on its own, outside any graph first:

In [ ]:
history = [HumanMessage(content="Can I return HT-1001?")]
update = [AIMessage(content="Let me check that for you.")]

add_messages(history, update)

Run the same graph again with nothing changed but the annotation added to the State:

In [ ]:
builder = StateGraph(ChatState)
builder.add_node("first", first)
builder.add_node("second", second)
builder.add_edge(START, "first")
builder.add_edge("first", "second")
builder.add_edge("second", END)

graph = builder.compile()

result = graph.invoke({"messages": [HumanMessage(content="hello")]})

for m in result["messages"]:
    print(f"{type(m).__name__:14s} {m.text}")

**Important note:**

If an incoming message carries the **same id** as one already in the list, `add_messages` replaces it rather
than appending. That is how a message is edited or removed.

## 5. `MessagesState`

A list of messages under a key called `messages` with the `add_messages` reducer is
so common that LangGraph ships it as built-in reducer:

```python
class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
```

That is the whole definition. We'll use it from here on.

In [ ]:
from langgraph.graph import MessagesState

class MessagesState(MessagesState):
    # Add any keys needed beyond messages, which is pre-built 
    pass

## 6. A graph that calls the model

Let's define a single-node graph with one task: hand the conversation to the model and append whatever comes back.

In [ ]:
def tool_calling_llm(state: MessagesState) -> dict:
    return {"messages": [model_with_tools.invoke(state["messages"])]}

#TODO: build START -> tool_calling_llm -> END

graph = builder.compile()

from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

Ask it something conversational and it behaves like a chatbot:

In [ ]:
result = graph.invoke({"messages": 
                       [HumanMessage(content="Hi, what can you help me with?")]})
result["messages"][-1].pretty_print()

Now ask it something that needs a tool:

In [ ]:
result = graph.invoke({"messages": [HumanMessage(content="Has order HT-1001 been delivered?")]})

for m in result["messages"]:
    m.pretty_print()

The graph ends with the model **asking** for `look_up_order` and no answer for the
customer. 

Nothing in this graph runs tools, so the request simply sits there.

That missing piece is the subject of the next notebook: a node that executes tool
calls, an edge that decides when to use it, and one more edge that turns the whole
thing into an agent.